In [0]:
# MAGIC %md ## 1.Parameters

In [0]:
dbutils.widgets.text("catalog", "mobility", "Catalog")
dbutils.widgets.text("month", "2024-01", "Month (YYYY-MM)")

In [0]:
CATALOG = dbutils.widgets.get("catalog")
MONTH = dbutils.widgets.get("month")

In [0]:
RAW = f"/Volumes/{CATALOG}/landing/raw"
YELLOW_PATH = f"{RAW}/yellow/yellow_tripdata_{MONTH}.parquet"
GREEN_PATH = f"{RAW}/green/green_tripdata_{MONTH}.parquet"
ZONE_LOOKUP_PATH = f"{RAW}/reference/taxi_zone_lookup.csv"
CBD_ZONES_PATH = f"{RAW}/reference/mta_cbd_taxi_zones.csv"

print(YELLOW_PATH, GREEN_PATH, ZONE_LOOKUP_PATH, CBD_ZONES_PATH, sep="\n")

2. Load raw files

In [0]:
from pyspark.sql import functions as F

yellow_raw=spark.read.parquet(YELLOW_PATH)
green_raw=spark.read.parquet(GREEN_PATH)

print(f"yellow rows: {yellow_raw.count():,}")
print(f"green rows: {green_raw.count():,}")



In [0]:
yellow_raw.printSchema()


In [0]:
green_raw.printSchema()

3. Schema differences between yellow and green

In [0]:
y_types = dict(yellow_raw.dtypes)
g_types = dict(green_raw.dtypes)

print("only in yellow:", sorted(set(y_types) - set(g_types)))
print("only in green:", sorted(set(g_types) - set(y_types)))



In [0]:
print("Same name, different type:")
for c in sorted(set(y_types) & set(g_types)):
    if y_types[c] != g_types[c]:
        print(f"  {c}: yellow={y_types[c]}, green={g_types[c]}")

4. Standardize

In [0]:
def standardize(df, service):
    """
    rename the service specific timestanp columns and tag the service
    """
    prefix = "tpep" if service == "yellow" else "lpep"
    return (
    df.withColumnRenamed(f"{prefix}_pickup_datetime", "pickup_ts")
          .withColumnRenamed(f"{prefix}_dropoff_datetime", "dropoff_ts")
          .withColumn("service_type", F.lit(service))
    )


In [0]:
trips = standardize(yellow_raw,"yellow").unionByName(standardize(green_raw,"green"),allowMissingColumns=True)
trips.printSchema()
trips.count()

In [0]:
from pyspark.sql.types import NumericType

numeric_cols = [f"{f.name}: {f.dataType.simpleString()}" for f in trips.schema.fields if isinstance(f.dataType, NumericType)]
print("Numeric columns:")
for c in numeric_cols:
    print(" ", c)

In [0]:
trips.createOrReplaceTempView("trips_profile")

5. Null counts per coloumn

In [0]:
null_counts = trips.groupBy("service_type").agg(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in trips.columns if c != "service_type"]
)
display(null_counts)

6. Numeric Distributions

In [0]:
numeric_cols = ["trip_distance", "fare_amount", "tip_amount", "tolls_amount",
                "total_amount", "passenger_count", "congestion_surcharge"]



In [0]:
for svc in ["yellow", "green"]:
    print(f"--- {svc} ---")
    display(
        trips.filter(F.col("service_type") == svc)
             .select(*numeric_cols)
             .summary("count", "min", "1%", "25%", "50%", "75%", "99%", "max")
    )

7. Candidate Data Quality Rules

In [0]:
month_start = f"DATE'{MONTH}-01'"
duration_min = "(unix_timestamp(dropoff_ts) - unix_timestamp(pickup_ts)) / 60.0"

In [0]:
# name -> (failure expression, proposed severity)
RULES = {
    "dropoff_not_after_pickup": ("dropoff_ts <= pickup_ts", "error"),
    "duration_under_1_min":     (f"{duration_min} < 1", "error"),
    "duration_over_240_min":    (f"{duration_min} > 240", "error"),
    "pickup_outside_month":     (f"pickup_ts < {month_start} - INTERVAL 1 DAY "
                                 f"OR pickup_ts >= add_months({month_start}, 1) + INTERVAL 1 DAY", "error"),
    "distance_not_positive":    ("trip_distance <= 0", "error"),
    "distance_over_100_mi":     ("trip_distance > 100", "error"),
    "negative_fare":            ("fare_amount < 0", "error"),
    "negative_total":           ("total_amount < 0", "error"),
    "speed_over_80_mph":        (f"try_divide(trip_distance, {duration_min} / 60.0) > 80", "error"),
    "pickup_zone_unknown":      ("PULocationID IS NULL OR PULocationID NOT BETWEEN 1 AND 263", "error"),
    "dropoff_zone_unknown":     ("DOLocationID IS NULL OR DOLocationID NOT BETWEEN 1 AND 263", "error"),
    "passenger_count_null":     ("passenger_count IS NULL", "warn"),
    "passenger_count_0_or_gt6": ("passenger_count = 0 OR passenger_count > 6", "warn"),
    "payment_type_invalid":     ("payment_type IS NULL OR payment_type NOT BETWEEN 0 AND 6", "warn"),
    "ratecode_invalid":         ("RatecodeID IS NULL OR RatecodeID NOT IN (1, 2, 3, 4, 5, 6)", "warn"),
    "tip_exceeds_fare":         ("tip_amount > fare_amount AND fare_amount > 0", "warn"),
}

In [0]:
#count failures for every rule at once.
agg_exprs = [F.count(F.lit(1)).alias("total_rows")] + [
    F.sum(F.coalesce(F.expr(expr), F.lit(False)).cast("int")).alias(name)
    for name, (expr, _) in RULES.items()
]

In [0]:
wide = trips.groupBy("service_type").agg(*agg_exprs)

# Reshape to one row per (service, rule).
stack_expr = "stack({n}, {pairs}) AS (rule, failed_rows)".format(
    n=len(RULES), pairs=", ".join(f"'{r}', {r}" for r in RULES)
)
severity = spark.createDataFrame([(r, s) for r, (_, s) in RULES.items()], "rule string, severity string")

In [0]:
dq_profile = (
    wide.selectExpr("service_type", "total_rows", stack_expr)
        .join(severity, "rule")
        .withColumn("failed_pct", F.round(F.col("failed_rows") / F.col("total_rows") * 100, 3))
        .withColumn("month", F.lit(MONTH))
        .withColumn("profiled_at", F.current_timestamp())
        .select("month", "service_type", "rule", "severity", "total_rows", "failed_rows", "failed_pct", "profiled_at")
        .orderBy("service_type", F.desc("failed_rows"))
)
display(dq_profile)

In [0]:
(dq_profile.write
    .mode("overwrite")
    .option("replaceWhere", f"month = '{MONTH}'")
    .saveAsTable(f"{CATALOG}.ops.profile_results"))


In [0]:
error_any = " OR ".join(f"coalesce(({e}), false)" for e, s in RULES.values() if s == "error")
display(
    trips.groupBy("service_type").agg(
        F.count(F.lit(1)).alias("total_rows"),
        F.sum(F.expr(error_any).cast("int")).alias("rows_failing_any_error_rule"),
    ).withColumn("quarantine_pct", F.round(F.col("rows_failing_any_error_rule") / F.col("total_rows") * 100, 3)))

In [0]:
display(
    trips.filter(F.expr(RULES["pickup_outside_month"][0]))
         .select("service_type", "pickup_ts", "dropoff_ts", "trip_distance", "total_amount")
         .orderBy("pickup_ts")
         .limit(20)
)

8. Trips per pickup day (stray dates and gaps)

In [0]:
display(
    trips.groupBy("service_type", F.to_date("pickup_ts").alias("pickup_date"))
         .count()
         .orderBy("pickup_date")
)

9. reference files

In [0]:
zones = (spark.read.option("header", True).option("inferSchema", True).csv(ZONE_LOOKUP_PATH))
zones.printSchema()
print(f"zone rows: {zones.count()}")
display(zones.groupBy("Borough").count().orderBy(F.desc("count")))

In [0]:
display(dbutils.fs.ls("/Volumes/mobility/landing/raw/reference/"))

In [0]:
dbutils.fs.mv(
    "/Volumes/mobility/landing/raw/reference/MTA_Central_Business_District_Taxi_Zones_20261120.csv",
    "/Volumes/mobility/landing/raw/reference/mta_cbd_taxi_zones.csv",
)
display(dbutils.fs.ls("/Volumes/mobility/landing/raw/reference/"))

In [0]:
cbd = (spark.read
       .option("header", True)
       .option("multiLine", True)
       .option("escape", '"')
       .csv(CBD_ZONES_PATH))
print("MTA CBD columns:", cbd.columns)
print(f"CBD zone rows: {cbd.count()}")



In [0]:
id_candidates = [c for c in cbd.columns if "location" in c.lower() and "id" in c.lower()] \
    or [c for c in cbd.columns if "zone" in c.lower() and "id" in c.lower()]
CBD_ID_COL = id_candidates[0] if id_candidates else None
print("Using CBD zone ID column:", CBD_ID_COL)

In [0]:
print(cbd.columns)

In [0]:
CBD_COL = "Taxi Zone"

cbd_vals = cbd.select(F.trim(F.col(CBD_COL)).alias("cbd_zone")).distinct()
display(cbd_vals.orderBy("cbd_zone"))

# Are the values zone numbers or zone names?
n_total = cbd_vals.count()
n_numeric = cbd_vals.filter(F.expr("try_cast(cbd_zone AS INT) IS NOT NULL")).count()
print(f"{n_numeric} of {n_total} values are numeric")

zones_n = zones.withColumn("zone_trim", F.trim("Zone"))

if n_numeric == n_total:
    # Values are LocationIDs
    cbd_ids = cbd_vals.select(F.col("cbd_zone").cast("int").alias("LocationID"))
    unmatched = cbd_ids.join(zones_n, "LocationID", "left_anti")
    matched = cbd_ids.join(zones_n, "LocationID")
else:
    # Values are zone names
    cond = cbd_vals.cbd_zone == zones_n.zone_trim
    unmatched = cbd_vals.join(zones_n, cond, "left_anti")
    matched = cbd_vals.join(zones_n, cond)

print(f"CBD zones not found in taxi_zone_lookup: {unmatched.count()}")
display(unmatched)

cbd_zone_ids = matched.select("LocationID", "Borough", "Zone", "service_zone").orderBy("LocationID")
display(cbd_zone_ids)
print(f"CBD zones matched: {cbd_zone_ids.count()}")

In [0]:

CBD_ID_COL = "Taxi Zone"

cbd_ids = (cbd.select(F.trim(F.col(CBD_ID_COL)).cast("int").alias("LocationID"))
              .distinct())

unmatched = cbd_ids.join(zones.select("LocationID"), "LocationID", "left_anti")
n_unmatched = unmatched.count()
print(f"CBD zones not found in taxi_zone_lookup: {n_unmatched}  (expect 0)")

cbd_zone_ids = (cbd_ids.join(zones, "LocationID")
                       .select("LocationID", "Borough", "Zone", "service_zone")
                       .orderBy("LocationID"))
display(cbd_zone_ids)

n_cbd = cbd_zone_ids.count()
n_not_manhattan = cbd_zone_ids.filter(F.col("Borough") != "Manhattan").count()
print(f"CBD zones matched: {n_cbd}; outside Manhattan: {n_not_manhattan}  (expect 0)")

# assertions
assert n_unmatched == 0, "Some CBD zones are missing from taxi_zone_lookup"
assert n_not_manhattan == 0, "CBD zones should all be in Manhattan"

10. Summary

In [0]:
rows = dq_profile.orderBy("service_type", "rule").collect()
lines = ["| Service | Rule | Severity | Failed rows | Failed % |", "| --- | --- | --- | --- | --- |"]
lines += [f"| {r.service_type} | {r.rule} | {r.severity} | {r.failed_rows:,} | {r.failed_pct} |" for r in rows]
print(f"Profiled month: {MONTH}\n")
print("\n".join(lines))